# ICT324 · Milestone 3 · Traffy Fondue Insight
**Team Ctrl C + Ctrl V:** Aung Myo (6702955), Aye Chan Ko (6703266), Phoo Myat May Zaw (6703013)

**What this notebook does (Milestone 3: data cleaned + first results)**
1. Downloads the official Traffy Fondue CSV and describes it (missing values, duplicates, outliers)
2. Cleans the data and writes a **cleaning log** (what we changed and why)
3. **What:** text labelling with a majority baseline + Logistic Regression, Linear SVM and Random Forest, **10-fold stratified CV, same folds (seed 1992)**, reporting precision, recall, F1 and AUC as mean ± std
4. **Translates the Thai complaints to English** (NLLB-200), tests Thai vs. English models, lists top English keywords per type, and exports a readable file
5. **Where:** first hotspot map with DBSCAN + heat map
6. **When:** season, month, weekday and hour charts
7. **Why:** daily rainfall vs. flooding complaints
8. Saves every table, chart and map into one zip for the project page

**How to run:** Runtime → Change runtime type → **T4 GPU** (for the translation), then Runtime → Run all. With the default settings it takes roughly 20–40 minutes; lower `SAMPLE_N` if it is too slow.

In [ ]:
# ---------- setup ----------
!pip -q install iterative-stratification==0.1.9 sentencepiece

import os, re, time, json, warnings, zipfile
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

CSV_URL   = "https://publicapi.traffy.in.th/dump-csv-chadchart/bangkok_traffy.csv"
CSV_PATH  = "bangkok_traffy.csv"
OUT       = "m3_outputs"            # everything we save goes here
SEED      = 1992                    # same seed as Lab 4
N_FOLDS   = 10                      # 10-fold stratified CV (Ajarn's rule)
SAMPLE_N  = 30000                   # complaints used for the text models (lower = faster)
TOP_K     = 10                      # number of problem types the text model predicts
TRAIN_START, TRAIN_END = "2022-06-01", "2025-10-31"   # labelled period
BKK_BOX   = dict(lat=(13.45, 14.05), lon=(100.30, 100.95))  # rough Bangkok bounding box
TRANSLATE = True                    # Thai -> English with NLLB-200 (needs Runtime > Change runtime type > T4 GPU)
MT_MODEL  = "facebook/nllb-200-distilled-600M"
MT_BATCH  = 32

os.makedirs(OUT, exist_ok=True)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 80)

# Thai problem types -> English (for charts; matplotlib cannot draw Thai by default)
TYPE_EN = {"ถนน":"Road","ทางเท้า":"Sidewalk","ความปลอดภัย":"Safety","แสงสว่าง":"Street lighting",
           "กีดขวาง":"Obstruction","ความสะอาด":"Cleanliness","ร้องเรียน":"General complaint","จราจร":"Traffic",
           "ป้าย":"Signs","น้ำท่วม":"Flooding","ท่อระบายน้ำ":"Drainage","ต้นไม้":"Trees","คลอง":"Canal",
           "สะพาน":"Bridge","สายไฟ":"Power lines","เสียงรบกวน":"Noise","คนจรจัด":"Homeless people",
           "ห้องน้ำ":"Public toilets","ป้ายจราจร":"Traffic signs","PM2.5":"PM2.5","การเดินทาง":"Transport",
           "สอบถาม":"Inquiry","เสนอแนะ":"Suggestion"}
en = lambda t: TYPE_EN.get(t, t)
# 50 Bangkok districts -> English names (fixed list, so names are never mistranslated)
DISTRICT_EN = {"พระนคร":"Phra Nakhon","ดุสิต":"Dusit","หนองจอก":"Nong Chok","บางรัก":"Bang Rak","บางเขน":"Bang Khen",
 "บางกะปิ":"Bang Kapi","ปทุมวัน":"Pathum Wan","ป้อมปราบศัตรูพ่าย":"Pom Prap Sattru Phai","พระโขนง":"Phra Khanong",
 "มีนบุรี":"Min Buri","ลาดกระบัง":"Lat Krabang","ยานนาวา":"Yan Nawa","สัมพันธวงศ์":"Samphanthawong","พญาไท":"Phaya Thai",
 "ธนบุรี":"Thon Buri","บางกอกใหญ่":"Bangkok Yai","ห้วยขวาง":"Huai Khwang","คลองสาน":"Khlong San","ตลิ่งชัน":"Taling Chan",
 "บางกอกน้อย":"Bangkok Noi","บางขุนเทียน":"Bang Khun Thian","ภาษีเจริญ":"Phasi Charoen","หนองแขม":"Nong Khaem",
 "ราษฎร์บูรณะ":"Rat Burana","บางพลัด":"Bang Phlat","ดินแดง":"Din Daeng","บึงกุ่ม":"Bueng Kum","สาทร":"Sathon",
 "บางซื่อ":"Bang Sue","จตุจักร":"Chatuchak","บางคอแหลม":"Bang Kho Laem","ประเวศ":"Prawet","คลองเตย":"Khlong Toei",
 "สวนหลวง":"Suan Luang","จอมทอง":"Chom Thong","ดอนเมือง":"Don Mueang","ราชเทวี":"Ratchathewi","ลาดพร้าว":"Lat Phrao",
 "วัฒนา":"Watthana","บางแค":"Bang Khae","หลักสี่":"Lak Si","สายไหม":"Sai Mai","คันนายาว":"Khan Na Yao",
 "สะพานสูง":"Saphan Sung","วังทองหลาง":"Wang Thonglang","คลองสามวา":"Khlong Sam Wa","บางนา":"Bang Na",
 "ทวีวัฒนา":"Thawi Watthana","ทุ่งครุ":"Thung Khru","บางบอน":"Bang Bon"}
den = lambda d: DISTRICT_EN.get(str(d).strip(), d)
SEASON = {3:"Hot",4:"Hot",5:"Hot",6:"Rainy",7:"Rainy",8:"Rainy",9:"Rainy",10:"Rainy",11:"Cool",12:"Cool",1:"Cool",2:"Cool"}
print("Ready. Outputs go to:", OUT)

## 1. Load the data

In [ ]:
if not os.path.exists(CSV_PATH):
    !wget -q --show-progress -O {CSV_PATH} {CSV_URL}

COLS = ["ticket_id","type","organization","comment","coords","district","subdistrict",
        "timestamp","state","last_activity","count_reopen","photo"]
raw = pd.read_csv(CSV_PATH, usecols=lambda c: c in COLS, dtype=str, low_memory=False)
print(f"Rows: {len(raw):,}   Columns: {raw.shape[1]}")
raw.head(3)

## 2. Data understanding: missing values, duplicates, states

In [ ]:
desc = pd.DataFrame({
    "missing": raw.isna().sum(),
    "missing %": (raw.isna().mean()*100).round(2),
    "unique values": raw.nunique(),
})
desc.to_csv(f"{OUT}/01_missing_values.csv")
display(desc)

print("Duplicate ticket_id rows:", f"{raw['ticket_id'].duplicated().sum():,}")
print("\nState counts:"); print(raw["state"].value_counts(dropna=False).head(10))

## 3. Cleaning (with a log of what we changed)

In [ ]:
log = []
def step(df, what, why):
    log.append({"step": what, "rows after": len(df), "why": why}); return df

df = step(raw.copy(), "Start", "Raw download")
df = step(df.drop_duplicates("ticket_id"), "Drop duplicate tickets", "Same complaint counted twice")

# time: report time and last update, converted to Bangkok time
for c in ["timestamp","last_activity"]:
    df[c] = pd.to_datetime(df[c], errors="coerce", utc=True, format="mixed").dt.tz_convert("Asia/Bangkok")
df = step(df.dropna(subset=["timestamp"]), "Drop rows without a report time", "Cannot place them in time")
df = step(df[df["timestamp"] >= pd.Timestamp(TRAIN_START, tz="Asia/Bangkok")],
          "Drop early test records (before June 2022)", "System launched in mid-2022; earlier rows are tests")

# location: coords are 'lon,lat'
ll = df["coords"].str.split(",", expand=True)
df["lon"] = pd.to_numeric(ll[0], errors="coerce"); df["lat"] = pd.to_numeric(ll[1], errors="coerce")
inside = df["lat"].between(*BKK_BOX["lat"]) & df["lon"].between(*BKK_BOX["lon"])
df = step(df[inside], "Drop GPS points outside Bangkok", "Wrong or missing location")

# problem types: '{ถนน,ทางเท้า}' -> ['ถนน','ทางเท้า']
df["types"] = (df["type"].fillna("").str.strip("{}").str.split(",")
               .apply(lambda xs: [x.strip() for x in xs if x.strip()]))
df["labelled"] = df["types"].str.len() > 0
df["comment"] = df["comment"].fillna("").str.replace(r"http\S+", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()

# time features
df["year"], df["month"] = df["timestamp"].dt.year, df["timestamp"].dt.month
df["weekday"], df["hour"] = df["timestamp"].dt.dayofweek, df["timestamp"].dt.hour
df["season"] = df["month"].map(SEASON)
df["district_clean"] = df["district"].fillna("").str.strip().str.replace(r"^เขต\s*", "", regex=True)
df["district_en"] = df["district_clean"].map(DISTRICT_EN)            # NaN if not one of the 50 Bangkok districts
bad_d = df.loc[df["district_en"].isna(), "district"].value_counts()
log.append({"step": f"Check district names: {df['district_en'].notna().sum():,} rows match the 50 Bangkok districts",
            "rows after": len(df), "why": f"{len(bad_d)} other spellings/areas kept but excluded from district analysis"})
df["date"] = df["timestamp"].dt.date

# fixing time for finished cases + outlier flags
df["finished"] = df["state"].eq("เสร็จสิ้น")
df["fix_days"] = (df["last_activity"] - df["timestamp"]).dt.total_seconds() / 86400
df.loc[~df["finished"], "fix_days"] = np.nan
df["fix_outlier"] = df["fix_days"].lt(0) | df["fix_days"].gt(365)
df["count_reopen"] = pd.to_numeric(df["count_reopen"], errors="coerce").fillna(0)
df["has_photo"] = df["photo"].notna()

clean_log = pd.DataFrame(log); clean_log["rows removed"] = (-clean_log["rows after"].diff()).fillna(0).astype(int)
print("District names not matching the 50 Bangkok districts (top 10):"); print(bad_d.head(10))
clean_log.to_csv(f"{OUT}/02_cleaning_log.csv", index=False)
display(clean_log)
print(f"Clean rows: {len(df):,}   labelled: {df['labelled'].mean()*100:.1f}%   finished: {df['finished'].mean()*100:.1f}%")

## 4. Describe the clean data: outliers and coverage

In [ ]:
fd = df.loc[df["finished"] & ~df["fix_outlier"], "fix_days"]
q1, q3 = fd.quantile([.25,.75]); iqr = q3 - q1
outl = pd.DataFrame({
    "fix_days (finished, 0-365)": fd.describe().round(2),
}).T
outl["IQR outliers (> Q3+1.5·IQR)"] = int((fd > q3 + 1.5*iqr).sum())
outl["flagged <0 or >365 days"] = int(df["fix_outlier"].sum())
outl.to_csv(f"{OUT}/03_fix_days_outliers.csv"); display(outl)

empty_by_year = (~df["labelled"]).groupby(df["year"]).mean().mul(100).round(1).rename("empty type %")
empty_by_year.to_csv(f"{OUT}/04_empty_type_by_year.csv"); display(empty_by_year.to_frame())

## 5. What: label the problem type from the complaint text
- Labelled period June 2022 – October 2025, complaints with text, top `TOP_K` types (multi-label: one complaint can have several types)
- Text features: TF-IDF on **character n-grams (2–4)**, which works for Thai without a word tokenizer
- TF-IDF is fitted **inside each fold**, so the test fold never leaks into training
- **Same 10 stratified folds and seed 1992 for every model** (multi-label stratified K-fold)

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, hamming_loss

lab = df[df["labelled"] & (df["timestamp"] <= pd.Timestamp(TRAIN_END+" 23:59", tz="Asia/Bangkok")) & (df["comment"].str.len() >= 5)]
top = pd.Series([t for ts in lab["types"] for t in ts]).value_counts().head(TOP_K).index.tolist()
lab = lab[lab["types"].apply(lambda ts: any(t in top for t in ts))]
n_before = len(lab)
lab = lab.assign(_norm=lab["comment"].str.lower().str.replace(r"\s+", "", regex=True)).drop_duplicates("_norm").drop(columns="_norm")
DUP_REMOVED = n_before - len(lab)
print(f"Removed {DUP_REMOVED:,} copy-pasted duplicate complaints ({DUP_REMOVED/n_before*100:.1f}%) so the same text cannot be in both train and test folds")
lab = lab.sample(min(SAMPLE_N, len(lab)), random_state=SEED)
X = lab["comment"].values
Y = np.array([[int(t in ts) for t in top] for ts in lab["types"]])
LABELS = [en(t) for t in top]
print(f"Text set: {len(lab):,} complaints  |  labels: {LABELS}")
print("Share of complaints with each label (%):", dict(zip(LABELS, (Y.mean(0)*100).round(1))))

In [ ]:
def tfidf(maxf): return TfidfVectorizer(analyzer="char_wb", ngram_range=(2,4), min_df=3, max_features=maxf, sublinear_tf=True)

MODELS = {   # ordered from simplest to most complex
  "Baseline (majority)": (Pipeline([("tfidf", tfidf(1000)), ("clf", OneVsRestClassifier(DummyClassifier(strategy="most_frequent")))]),
                          "predicts the most common value for every label"),
  "Logistic Regression": (Pipeline([("tfidf", tfidf(50000)), ("clf", OneVsRestClassifier(LogisticRegression(C=4, solver="liblinear", class_weight="balanced", max_iter=1000)))]),
                          "TF-IDF char 2-4, 50k features; C=4, liblinear, class_weight=balanced"),
  "Linear SVM":          (Pipeline([("tfidf", tfidf(50000)), ("clf", OneVsRestClassifier(LinearSVC(C=0.5, class_weight="balanced")))]),
                          "TF-IDF char 2-4, 50k features; C=0.5, class_weight=balanced"),
  "Random Forest":       (Pipeline([("tfidf", tfidf(5000)),  ("clf", RandomForestClassifier(n_estimators=100, min_samples_leaf=2, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED))]),
                          "TF-IDF char 2-4, 5k features; 100 trees, min_samples_leaf=2, balanced_subsample"),
}

def scores(model, Xt):
    clf = model.named_steps["clf"]
    if hasattr(clf, "decision_function") and not isinstance(clf, RandomForestClassifier):
        try: return model.decision_function(Xt)
        except Exception: pass
    p = model.predict_proba(Xt)
    if isinstance(p, list):   # Random Forest multi-label: one (n,2) array per label
        return np.column_stack([pi[:,1] if pi.shape[1] > 1 else np.zeros(len(Xt)) for pi in p])
    return p

def run_cv(models, Xs, Ys, folds, tag=""):
    rows, oof = [], {}
    for name, (model, params) in models.items():
        t0, fold_res, pred_all = time.time(), [], np.zeros_like(Ys)
        for k, (tr, te) in enumerate(folds):
            model.fit(Xs[tr], Ys[tr])
            yp, ys = model.predict(Xs[te]), scores(model, Xs[te])
            pred_all[te] = yp
            try: auc = roc_auc_score(Ys[te], ys, average="macro")
            except ValueError: auc = np.nan
            fold_res.append(dict(
                macro_f1=f1_score(Ys[te], yp, average="macro", zero_division=0),
                micro_f1=f1_score(Ys[te], yp, average="micro", zero_division=0),
                precision=precision_score(Ys[te], yp, average="macro", zero_division=0),
                recall=recall_score(Ys[te], yp, average="macro", zero_division=0),
                auc=auc, hamming=hamming_loss(Ys[te], yp)))
        fr = pd.DataFrame(fold_res); oof[name] = pred_all
        rows.append({"Model": name, **{f"{c} mean": fr[c].mean() for c in fr}, **{f"{c} std": fr[c].std() for c in fr},
                     "Parameters": params, "Time (s)": round(time.time()-t0)})
        print(f"{tag}{name:22s} macro F1 {fr.macro_f1.mean():.3f} ± {fr.macro_f1.std():.3f}   AUC {fr.auc.mean():.3f}   ({time.time()-t0:.0f}s)")
    return pd.DataFrame(rows), oof

folds = list(MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X, Y))
res, oof = run_cv(MODELS, X, Y, folds)
res.to_csv(f"{OUT}/05_text_model_results_raw.csv", index=False)

In [ ]:
# Results table in Lab 4 style: mean ± std over the 10 folds (percent)
fmt = lambda m, s: f"{m*100:.1f} ± {s*100:.1f}"
table = pd.DataFrame({
    "Model": res["Model"],
    "Macro F1 %": [fmt(m,s) for m,s in zip(res["macro_f1 mean"], res["macro_f1 std"])],
    "Precision %": [fmt(m,s) for m,s in zip(res["precision mean"], res["precision std"])],
    "Recall %": [fmt(m,s) for m,s in zip(res["recall mean"], res["recall std"])],
    "AUC %": [fmt(m,s) for m,s in zip(res["auc mean"], res["auc std"])],
    "Micro F1 %": [fmt(m,s) for m,s in zip(res["micro_f1 mean"], res["micro_f1 std"])],
    "Hamming loss": [f"{m:.3f}" for m in res["hamming mean"]],
    "Parameters": res["Parameters"],
})
table.to_csv(f"{OUT}/06_text_model_results_table.csv", index=False); display(table)

# Simplest model that is "good enough": within one std of the best macro F1
best = res.loc[res["macro_f1 mean"].idxmax()]
ok = res[res["macro_f1 mean"] >= best["macro_f1 mean"] - best["macro_f1 std"]]
simplest = ok.iloc[0]["Model"]   # MODELS is ordered simplest -> most complex
verdict = (f"Best macro F1: {best['Model']} ({best['macro_f1 mean']*100:.1f}%). "
           f"Simplest model within one std of it: {simplest}.")
print(verdict)

# chart
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(res["Model"], res["macro_f1 mean"]*100, yerr=res["macro_f1 std"]*100, capsize=6, color=["#9AA3BD","#FFC23D","#FF8A3D","#3CCB8A"])
ax.set_ylabel("Macro F1 (%)"); ax.set_title(f"Text labelling, {N_FOLDS}-fold stratified CV (mean ± std)")
for i, v in enumerate(res["macro_f1 mean"]*100): ax.text(i, v+1, f"{v:.1f}", ha="center")
plt.tight_layout(); plt.savefig(f"{OUT}/07_text_model_macro_f1.png", dpi=150); plt.show()

In [ ]:
# Per problem type: F1 from out-of-fold predictions (which types are hard?)
per = pd.DataFrame({m: f1_score(Y, oof[m], average=None, zero_division=0) for m in MODELS}, index=LABELS).mul(100).round(1)
per.insert(0, "share of complaints %", (Y.mean(0)*100).round(1))
per.to_csv(f"{OUT}/08_f1_per_problem_type.csv"); display(per)

## 6. Thai to English: machine translation of the text sample
We cannot read Thai, so we translate every complaint in the text sample with **NLLB-200** (Meta's open translation model).
- Needs a GPU: **Runtime → Change runtime type → T4 GPU**. About 15–30 minutes for 30,000 complaints.
- Progress is saved every 1,000 complaints in `translation_cache.csv`, so a re-run continues where it stopped. **Tip:** upload last run's `translation_cache.csv` into the `m3_outputs` folder before running to skip complaints already translated.
- Without a GPU it translates only 1,500 complaints, so you can still test the steps.
- The models still learn from the **original Thai**. The English is for **us to read** and for the **Thai vs. English test** (RQ1).

In [ ]:
import torch
try:
    from transformers.utils import logging as hf_logging; hf_logging.set_verbosity_error()
except Exception: pass
lab = lab.copy(); lab["types_en"] = lab["types"].apply(lambda ts: ", ".join(en(t) for t in ts))
cache_path = f"{OUT}/translation_cache.csv"
cache = pd.read_csv(cache_path) if os.path.exists(cache_path) else pd.DataFrame(columns=["ticket_id","comment_en"])

def load_translator():
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(MT_MODEL, src_lang="tha_Thai")
    mt = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL, torch_dtype=torch.float16 if dev == "cuda" else torch.float32).to(dev).eval()
    eng = tok.convert_tokens_to_ids("eng_Latn")
    def translate(texts):
        out = []
        for i in range(0, len(texts), MT_BATCH):
            enc = tok(list(texts[i:i+MT_BATCH]), return_tensors="pt", padding=True, truncation=True, max_length=200).to(dev)
            with torch.no_grad():
                gen = mt.generate(**enc, forced_bos_token_id=eng, max_new_tokens=160, num_beams=1)
            out += tok.batch_decode(gen, skip_special_tokens=True)
        return out
    return translate, dev

if TRANSLATE:
    n_tr = len(lab) if torch.cuda.is_available() else min(1500, len(lab))
    if not torch.cuda.is_available():
        print("No GPU found. Translating only 1,500 complaints. For all of them: Runtime > Change runtime type > T4 GPU.")
    todo = lab.iloc[:n_tr]; todo = todo[~todo["ticket_id"].isin(set(cache["ticket_id"]))]
    if len(todo):
        translate, dev = load_translator()
        todo = todo.assign(_len=todo["comment"].str.len()).sort_values("_len")   # similar lengths batch faster
        t0 = time.time()
        for i in range(0, len(todo), 1000):
            part = todo.iloc[i:i+1000]
            cache = pd.concat([cache, pd.DataFrame({"ticket_id": part["ticket_id"].values,
                                                     "comment_en": translate(part["comment"].tolist())})])
            cache.to_csv(cache_path, index=False)
            print(f"translated {min(i+1000, len(todo)):,}/{len(todo):,}  ({time.time()-t0:.0f}s)")
lab_en = lab.merge(cache.drop_duplicates("ticket_id"), on="ticket_id", how="left")
print(f"English text ready for {lab_en['comment_en'].notna().sum():,} of {len(lab_en):,} complaints")
display(lab_en[["district_en","types_en","comment","comment_en"]].head(8))

In [ ]:
# Thai vs. English: same rows, same folds. English uses word TF-IDF (1-2 words); Thai uses character n-grams.
def tfidf_en(maxf): return TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2, max_features=maxf, sublinear_tf=True, stop_words="english")
MODELS_EN = {
  "Logistic Regression": (Pipeline([("tfidf", tfidf_en(50000)), ("clf", OneVsRestClassifier(LogisticRegression(C=4, solver="liblinear", class_weight="balanced", max_iter=1000)))]),
                          "English word TF-IDF 1-2, 50k; C=4, liblinear, balanced"),
  "Linear SVM":          (Pipeline([("tfidf", tfidf_en(50000)), ("clf", OneVsRestClassifier(LinearSVC(C=0.5, class_weight="balanced")))]),
                          "English word TF-IDF 1-2, 50k; C=0.5, balanced"),
}
mask = lab_en["comment_en"].notna().values
if mask.sum() >= 200:
    Xen, Xth, Ysub = lab_en.loc[mask, "comment_en"].fillna("").values, X[mask], Y[mask]
    if mask.all():
        folds_sub, res_th = folds, res[res["Model"].isin(MODELS_EN)]
    else:
        folds_sub = list(MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(Xth, Ysub))
        res_th, _ = run_cv({k: MODELS[k] for k in MODELS_EN}, Xth, Ysub, folds_sub, tag="[Thai]    ")
    res_en, oof_en = run_cv(MODELS_EN, Xen, Ysub, folds_sub, tag="[English] ")
    cmp = pd.DataFrame({"Model": res_en["Model"],
        "Thai macro F1 %": (res_th.set_index("Model").loc[res_en["Model"], "macro_f1 mean"].values*100).round(1),
        "English macro F1 %": (res_en["macro_f1 mean"].values*100).round(1),
        "English std %": (res_en["macro_f1 std"].values*100).round(1)})
    cmp["Difference (English - Thai)"] = (cmp["English macro F1 %"] - cmp["Thai macro F1 %"]).round(1)
    cmp.to_csv(f"{OUT}/09_thai_vs_english.csv", index=False); display(cmp)
    print("If the difference is within about one std, translation keeps enough meaning for labelling.")
else:
    cmp = None; print("Not enough translated complaints for the Thai vs. English test.")

In [ ]:
# What the model reads, in English: top words for each problem type (English Logistic Regression)
if cmp is not None:
    m = MODELS_EN["Logistic Regression"][0].fit(Xen, Ysub)
    vocab = np.array(m.named_steps["tfidf"].get_feature_names_out())
    kw = pd.DataFrame({LABELS[j]: vocab[np.argsort(est.coef_[0])[::-1][:10]]
                       for j, est in enumerate(m.named_steps["clf"].estimators_)})
    kw.index = [f"#{i+1}" for i in range(len(kw))]
    kw.to_csv(f"{OUT}/10_top_english_keywords_per_type.csv"); display(kw.T)

In [ ]:
# A file we can read: every translated complaint, the user's types, the model's types, and whether they disagree
if cmp is not None:
    pred = oof["Logistic Regression"][mask]          # Thai model, out-of-fold (honest) predictions
    read = lab_en.loc[mask, ["ticket_id","date","district_en","types_en","comment","comment_en"]].copy()
    read["model_types"] = [", ".join(l for l, v in zip(LABELS, row) if v) or "-" for row in pred]
    read["agree"] = [(set(a.split(", ")) & set(LABELS)) == set(b.split(", ")) - {"-"} for a, b in zip(read["types_en"], read["model_types"])]
    read = read.rename(columns={"comment": "comment_th", "district_en": "district", "types_en": "user_types"})
    read.sort_values("agree").to_csv(f"{OUT}/11_translated_complaints_to_check.csv", index=False, encoding="utf-8-sig")
    print(f"Model and user agree on {read['agree'].mean()*100:.1f}% of complaints. Disagreements are listed first in 11_translated_complaints_to_check.csv.")
    display(read[~read["agree"]].head(10)[["district","user_types","model_types","comment_en"]])

## 7. Where: first hotspot map (DBSCAN + heat map)
DBSCAN with the haversine distance groups complaints that are close together. `eps` = 150 m, `min_samples` = 25.
Shown for **Road** and **Flooding** complaints in 2024 – Oct 2025.

In [ ]:
import folium
from folium.plugins import HeatMap
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

EPS_M, MIN_S, R_EARTH = 150, 25, 6_371_000          # DBSCAN: 150 m radius, at least 25 complaints
ESRI_GRAY = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}"
ESRI_STREET = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}"
recent = df[(df["timestamp"] >= pd.Timestamp("2024-01-01", tz="Asia/Bangkok")) &
            (df["timestamp"] <= pd.Timestamp(TRAIN_END + " 23:59", tz="Asia/Bangkok"))]

fmap = folium.Map(location=[13.76, 100.54], zoom_start=11, tiles=None)   # Esri: free, no API key, works in Colab
folium.TileLayer(ESRI_GRAY, attr="Tiles &copy; Esri", name="Light gray map").add_to(fmap)
folium.TileLayer(ESRI_STREET, attr="Tiles &copy; Esri", name="Street map").add_to(fmap)

hot_rows = []
for thai, color in [("ถนน", "#d9480f"), ("น้ำท่วม", "#1c7ed6")]:      # Road = orange, Flooding = blue
    pts = recent[recent["types"].apply(lambda ts: thai in ts)]
    pts = pts.sample(min(20000, len(pts)), random_state=SEED)
    if len(pts) < MIN_S:
        print(f"{en(thai)}: not enough points"); continue
    xy = np.radians(pts[["lat", "lon"]].values)
    labels_ = DBSCAN(eps=EPS_M / R_EARTH, min_samples=MIN_S, metric="haversine", algorithm="ball_tree").fit_predict(xy)
    pts = pts.assign(cluster=labels_); core = labels_ >= 0
    sil = np.nan
    if core.sum() > 50 and len(set(labels_[core])) > 1:
        idx = np.random.RandomState(SEED).choice(np.where(core)[0], min(4000, core.sum()), replace=False)
        sil = silhouette_score(xy[idx], labels_[idx], metric="haversine")
    g = (pts[core].groupby("cluster")
         .agg(complaints=("ticket_id", "size"), lat=("lat", "mean"), lon=("lon", "mean"),
              district=("district_en", lambda s: s.mode().iat[0] if len(s.mode()) else "?"),
              median_fix_days=("fix_days", "median"))
         .sort_values("complaints", ascending=False))
    g.insert(0, "type", en(thai)); hot_rows.append(g.head(15))
    print(f"{en(thai)}: {len(pts):,} points, {g.shape[0]} hotspots, {(~core).mean()*100:.0f}% noise, silhouette {sil:.2f}")
    heat = folium.FeatureGroup(name=f"{en(thai)} heat map", show=(thai == "ถนน"))
    HeatMap(pts[["lat", "lon"]].values.tolist(), radius=8, blur=10, min_opacity=0.3,
            gradient={0.2: "#2b83ba", 0.5: "#abdda4", 0.75: "#fdae61", 1.0: "#d7191c"}).add_to(heat)
    heat.add_to(fmap)
    spots = folium.FeatureGroup(name=f"{en(thai)} top hotspots")
    for _, r in g.head(15).iterrows():
        fix = f"{r.median_fix_days:.1f} days" if pd.notna(r.median_fix_days) else "n/a"
        folium.CircleMarker([r.lat, r.lon], radius=4 + np.sqrt(r.complaints) / 2, color=color, fill=True,
            fill_opacity=0.75, weight=2,
            popup=folium.Popup(f"<b>{en(thai)} hotspot</b><br>{r.complaints} complaints<br>District: {r.district}<br>Median fix: {fix}", max_width=220)
        ).add_to(spots)
    spots.add_to(fmap)

legend = ('<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;padding:8px 12px;'
          'border-radius:6px;box-shadow:0 2px 8px rgba(0,0,0,.25);font:13px Arial"><b>Top hotspots</b><br>'
          '<span style="color:#d9480f">&#9679;</span> Road &nbsp;<span style="color:#1c7ed6">&#9679;</span> Flooding<br>'
          '<span style="color:#666">Bigger circle = more complaints</span></div>')
fmap.get_root().html.add_child(folium.Element(legend))
folium.LayerControl(collapsed=False).add_to(fmap)
fmap.save(f"{OUT}/12_hotspot_map.html")
hot = pd.concat(hot_rows) if hot_rows else pd.DataFrame()
hot.to_csv(f"{OUT}/13_top_hotspots.csv"); display(hot)
fmap

## 8. When: season, month, weekday and hour

In [ ]:
# complaints per month
monthly = df.set_index("timestamp").resample("MS").size()
fig, ax = plt.subplots(figsize=(10,3.6)); monthly.plot(ax=ax, color="#FF8A3D", lw=2)
ax.set_title("Complaints reported per month"); ax.set_ylabel("Complaints"); ax.set_xlabel("")
plt.tight_layout(); plt.savefig(f"{OUT}/14_complaints_per_month.png", dpi=150); plt.show()

# Seasonal index: average complaints per month in a season / average per month overall (1.0 = normal).
# Fair comparison, because the rainy season has 5 months and the hot season only 3.
per = df[df["labelled"] & (df["timestamp"] <= pd.Timestamp(TRAIN_END+" 23:59", tz="Asia/Bangkok"))].copy()
per["ym"] = per["timestamp"].dt.tz_localize(None).dt.to_period("M")
ex = per.explode("types").reset_index(drop=True); ex = ex[ex["types"].isin(top)]
monthly_t = ex.groupby(["types", "ym"]).size().rename("n").reset_index()
monthly_t["season"] = monthly_t["ym"].dt.month.map(SEASON)
idx_t = (monthly_t.groupby(["types", "season"])["n"].mean() / monthly_t.groupby("types")["n"].mean()).unstack()[["Hot","Rainy","Cool"]]
idx_t.index = idx_t.index.map(en)
all_m = per.groupby("ym").size().rename("n").reset_index(); all_m["season"] = all_m["ym"].dt.month.map(SEASON)
idx_t.loc["All complaints"] = (all_m.groupby("season")["n"].mean() / all_m["n"].mean())[["Hot","Rainy","Cool"]].values
season_idx = idx_t.round(2).sort_values("Rainy", ascending=False)
season_idx.to_csv(f"{OUT}/15_seasonal_index.csv"); display(season_idx)
print("Read it like this: 1.30 in Rainy = 30% more complaints per month than normal in the rainy season.")
ax = season_idx.plot(kind="barh", figsize=(9,5.5), color=["#FF8A3D","#1c7ed6","#9AA3BD"])
ax.axvline(1, color="black", lw=1, ls="--"); ax.set_xlabel("Seasonal index (1.0 = average month)")
ax.set_title("Which problems rise in each season (per-month average, fair across seasons)")
plt.tight_layout(); plt.savefig(f"{OUT}/16_seasonal_index.png", dpi=150); plt.show()

# hour x weekday heat map
hw = pd.crosstab(df["weekday"], df["hour"]); hw.index = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
fig, ax = plt.subplots(figsize=(11,3.6)); im = ax.imshow(hw.values, aspect="auto", cmap="magma")
ax.set_yticks(range(7)); ax.set_yticklabels(hw.index); ax.set_xticks(range(0,24,2)); ax.set_xlabel("Hour (Bangkok time)")
ax.set_title("When complaints are reported"); plt.colorbar(im, label="complaints")
plt.tight_layout(); plt.savefig(f"{OUT}/17_hour_weekday_heatmap.png", dpi=150); plt.show()
hw.to_csv(f"{OUT}/18_hour_weekday_counts.csv")

## 9. Why: daily rainfall vs. flooding complaints
Daily rainfall for central Bangkok from the free Open-Meteo historical weather API. This shows a **link**, not proof of cause.

In [ ]:
try:
    url = ("https://archive-api.open-meteo.com/v1/archive?latitude=13.75&longitude=100.50"
           f"&start_date={TRAIN_START}&end_date={TRAIN_END}&daily=precipitation_sum&timezone=Asia%2FBangkok")
    import requests
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=90); r.raise_for_status(); w = r.json()["daily"]; break
        except Exception as err:
            print(f"Weather download attempt {attempt+1} failed: {err}"); time.sleep(5)
    else:
        raise RuntimeError("weather API not reachable")
    rain = pd.DataFrame({"date": pd.to_datetime(pd.Series(w["time"])).dt.date, "rain_mm": w["precipitation_sum"]})
    flood = (df[df["types"].apply(lambda ts: "น้ำท่วม" in ts)].groupby("date").size().rename("flood_complaints").reset_index())
    daily = rain.merge(flood, on="date", how="left").fillna({"flood_complaints": 0})
    rho = daily[["rain_mm","flood_complaints"]].corr(method="spearman").iloc[0,1]
    daily.to_csv(f"{OUT}/19_daily_rain_vs_flooding.csv", index=False)
    print(f"Spearman correlation, daily rain vs. flooding complaints: {rho:.2f}")
    heavy = daily["rain_mm"] >= 20
    print(f"Flooding complaints per day: {daily.loc[heavy,'flood_complaints'].mean():.1f} on heavy-rain days (>= 20 mm) "
          f"vs {daily.loc[~heavy,'flood_complaints'].mean():.1f} on other days")
    fig, ax = plt.subplots(figsize=(10,3.6)); d = daily.set_index(pd.to_datetime(daily["date"]))
    d["flood_complaints"].rolling(7).mean().plot(ax=ax, color="#1c7ed6", label="Flooding complaints (7-day avg)")
    ax2 = ax.twinx(); d["rain_mm"].rolling(7).mean().plot(ax=ax2, color="#9AA3BD", label="Rain mm (7-day avg)")
    ax.set_title(f"Rain and flooding complaints (Spearman rho = {rho:.2f})"); ax.legend(loc="upper left"); ax2.legend(loc="upper right")
    plt.tight_layout(); plt.savefig(f"{OUT}/20_rain_vs_flooding.png", dpi=150); plt.show()
except Exception as e:
    rho = np.nan; print("Rainfall step skipped:", e)

## 10. Summary for the project page, and download everything

In [ ]:
summary = f"""Milestone 3 summary (auto-generated)
- Data: {len(raw):,} raw rows -> {len(df):,} clean rows (June 2022 onward, inside Bangkok, no duplicates). See 02_cleaning_log.csv.
- Empty problem type: {(~df['labelled']).mean()*100:.1f}% of clean rows. Rows matched to the 50 Bangkok districts: {df['district_en'].notna().mean()*100:.1f}%.
- Text set: removed {DUP_REMOVED:,} copy-pasted duplicate complaints before sampling.
- Text labelling ({len(lab):,} complaints, {TOP_K} types, {N_FOLDS}-fold stratified CV, seed {SEED}):
""" + "\n".join(f"  - {r['Model']}: macro F1 {r['macro_f1 mean']*100:.1f} ± {r['macro_f1 std']*100:.1f}%, AUC {r['auc mean']*100:.1f}%" for _, r in res.iterrows()) + f"""
- {verdict}
- Thai vs. English (machine translated): """ + ("; ".join(f"{r['Model']} {r['Thai macro F1 %']}% vs {r['English macro F1 %']}%" for _, r in cmp.iterrows()) if cmp is not None else "not run") + f"""
- Hotspots: see 12_hotspot_map.html and 13_top_hotspots.csv.
- Seasonal index (Rainy): """ + ", ".join(f"{k} {v:.2f}" for k, v in season_idx["Rainy"].head(4).items()) + f"""
- Rain vs. flooding complaints (Spearman): {rho:.2f}
"""
open(f"{OUT}/00_summary.txt", "w").write(summary); print(summary)

with zipfile.ZipFile("m3_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT)): z.write(os.path.join(OUT, f), f)
try:
    from google.colab import files; files.download("m3_outputs.zip")
except Exception:
    print("Saved m3_outputs.zip")